In [4]:
from pathlib import Path
import pandas as pd

# Paths
output_dir = Path("../../Output/Player Archetype Analysis").resolve()
visual_dir = Path("../../visual/Player Archetype").resolve()
dossier_output_dir = (output_dir / "player_dossier_demo").resolve()
dossier_visual_dir = (visual_dir / "player_dossier_demo").resolve()

# Read CSVs
macro_summary = pd.read_csv(output_dir / "archetype_macro_summary_table.csv")
shot_summary = pd.read_csv(output_dir / "shot_style_cluster_summary.csv")
drift_table = pd.read_csv(output_dir / "player_identity_drift_table.csv")
comp_dossier = pd.read_csv(output_dir / "player_comp_dossier_table.csv")
final_profiles = pd.read_csv(output_dir / "final_player_archetype_profile_table.csv")
cohort_audit = pd.read_csv(output_dir / "cohort_join_audit.csv")
dossier_selection = pd.read_csv(dossier_output_dir / "report_ready_player_dossier_selection.csv")

# Coverage stats
n_candidates = len(cohort_audit)
n_fully_covered = cohort_audit["fully_covered_for_hybrid_pipeline"].fillna(False).sum()
coverage_pct = round(100 * n_fully_covered / n_candidates, 1)

# Drift counts
drift_counts = (
    drift_table
    .groupby("identity_drift_class", dropna=False)
    .size()
    .reset_index(name="players")
)
drift_counts["share"] = round(100 * drift_counts["players"] / drift_counts["players"].sum(), 1)

# Executive macro table
macro_table_exec = pd.DataFrame({
    "Macro role family": macro_summary["macro_archetype"],
    "Representative players": macro_summary["representative_players"],
    "Minutes": macro_summary["mean_min_mean"].round(1),
    "PTS/36": macro_summary["pts_per36_4yr_mean"].round(1),
    "REB/36": macro_summary["reb_per36_4yr_mean"].round(1),
    "AST/36": macro_summary["ast_per36_4yr_mean"].round(1),
})

# Executive shot-style table
shot_table_exec = pd.DataFrame({
    "Shot-style subtype": shot_summary["shot_style_subtype"],
    "Players": shot_summary["subtype_size"],
    "Avg. membership prob.": shot_summary["avg_membership_probability"].round(3),
    "Avg. entropy": shot_summary["avg_entropy"].round(3),
    "Style note": shot_summary["descriptive_style_notes"],
})

# Executive drift table
drift_table_exec = pd.DataFrame({
    "Drift class": drift_counts["identity_drift_class"],
    "Players": drift_counts["players"],
    "Share (%)": drift_counts["share"],
})

# Executive case-study table
case_players = ["Trae Young", "Kemba Walker", "Roy Hibbert", "Nikola Vučević", "Kyle O'Quinn"]

case_table_exec = (
    final_profiles.loc[final_profiles["PLAYER_NAME"].isin(case_players)]
    .assign(
        **{
            "Comp group median points": final_profiles.loc[
                final_profiles["PLAYER_NAME"].isin(case_players), "median_comp_group_points"
            ].round(1).values,
            "Comp group median minutes": final_profiles.loc[
                final_profiles["PLAYER_NAME"].isin(case_players), "median_comp_group_minutes"
            ].round(1).values,
        }
    )
    .loc[:, [
        "PLAYER_NAME",
        "hybrid_archetype_label",
        "identity_drift_class",
        "realistic_comp_list",
        "ceiling_comp_PLAYER_NAME",
        "Comp group median points",
        "Comp group median minutes",
    ]]
    .rename(columns={
        "PLAYER_NAME": "Player",
        "hybrid_archetype_label": "Hybrid archetype",
        "identity_drift_class": "Drift class",
        "realistic_comp_list": "Realistic comps",
        "ceiling_comp_PLAYER_NAME": "Ceiling analog",
    })
)

# Optional preview
print("n_candidates:", n_candidates)
print("n_fully_covered:", n_fully_covered)
print("coverage_pct:", coverage_pct)

print("\nmacro_table_exec")
print(macro_table_exec.head())

print("\nshot_table_exec")
print(shot_table_exec.head())

print("\ndrift_table_exec")
print(drift_table_exec.head())

print("\ncase_table_exec")
print(case_table_exec)

n_candidates: 1247
n_fully_covered: 1006
coverage_pct: 80.7

macro_table_exec
                  Macro role family  \
0       High-Usage Primary Creators   
1           Low-Usage Interior Bigs   
2      Perimeter Wings & Connectors   
3  Fringe / Low-Opportunity Players   
4   Scoring Bigs / Two-Way Forwards   

                              Representative players  Minutes  PTS/36  REB/36  \
0  Brandon Knight, Rodney Stuckey, Kemba Walker, ...     30.4    17.7     4.4   
1  Ed Davis, Jake Tsakalidis, Todd MacCulloch, Ja...     15.5    11.9     9.1   
2  Svi Mykhailiuk, Damyean Dotson, Wayne Ellingto...     18.3    12.7     4.6   
3  Nick Johnson, Damir Markota, Damien Inglis, An...      7.8    10.9     6.2   
4  Markieff Morris, Willie Cauley-Stein, Aaron Go...     26.9    16.1     8.1   

   AST/36  
0     5.3  
1     1.5  
2     3.2  
3     2.2  
4     2.1  

shot_table_exec
             Shot-style subtype  Players  Avg. membership prob.  Avg. entropy  \
0          Rim Pressure Driver

In [5]:
print(macro_table_exec.round(1).to_latex(index=False))

\begin{tabular}{llrrrr}
\toprule
Macro role family & Representative players & Minutes & PTS/36 & REB/36 & AST/36 \\
\midrule
High-Usage Primary Creators & Brandon Knight, Rodney Stuckey, Kemba Walker, Isaiah Thomas, Bradley Beal & 30.400000 & 17.700000 & 4.400000 & 5.300000 \\
Low-Usage Interior Bigs & Ed Davis, Jake Tsakalidis, Todd MacCulloch, Jason Smith, Trevor Booker & 15.500000 & 11.900000 & 9.100000 & 1.500000 \\
Perimeter Wings & Connectors & Svi Mykhailiuk, Damyean Dotson, Wayne Ellington, E'Twaun Moore, Timothe Luwawu-Cabarrot & 18.300000 & 12.700000 & 4.600000 & 3.200000 \\
Fringe / Low-Opportunity Players & Nick Johnson, Damir Markota, Damien Inglis, Antonis Fotsis, Kim English & 7.800000 & 10.900000 & 6.200000 & 2.200000 \\
Scoring Bigs / Two-Way Forwards & Markieff Morris, Willie Cauley-Stein, Aaron Gordon, Myles Turner, Marvin Williams & 26.900000 & 16.100000 & 8.100000 & 2.100000 \\
\bottomrule
\end{tabular}



In [7]:
print(
    shot_table_exec.round(1).to_latex(
        index=False,
        float_format=lambda x: f"{x:.1f}".rstrip("0").rstrip(".")
    )
)

\begin{tabular}{lrrrl}
\toprule
Shot-style subtype & Players & Avg. membership prob. & Avg. entropy & Style note \\
\midrule
Rim Pressure Drivers & 495 & 1 & 0 & Paint-oriented creators with heavy rim pressure and lower perimeter dependence. \\
Balanced Three-Level Scorers & 64 & 1 & 0.1 & Profiles with meaningful rim, midrange, and three-point balance. \\
Arc-Heavy Spacers & 161 & 1 & 0 & High-frequency perimeter shot makers and floor-spacing profiles. \\
Interior Finishers & 83 & 1 & 0 & Big-oriented finishing maps concentrated near the basket and paint. \\
Midrange Shot Creators & 168 & 1 & 0.1 & Self-created midrange-heavy profiles with intermediate-area usage. \\
\bottomrule
\end{tabular}



In [8]:
print(
    case_table_exec.round(1).to_latex(
        index=False,
        float_format=lambda x: f"{x:.1f}".rstrip("0").rstrip(".")
    )
)

\begin{tabular}{lllllrr}
\toprule
Player & Hybrid archetype & Drift class & Realistic comps & Ceiling analog & Comp group median points & Comp group median minutes \\
\midrule
Roy Hibbert & Scoring Bigs / Two-Way Forwards | Arc-Heavy Spacers & role_shifting_materially & Aaron Gordon, Will Barton, Joffrey Lauvergne, Skal Labissiere, Darnell Jackson & Manu Ginobili & 256 & 636.5 \\
Kemba Walker & High-Usage Primary Creators | Rim Pressure Drivers & role_shifting_materially & Isaiah Thomas, D'Angelo Russell, Marcus Smart, Paul George, Nickeil Alexander-Walker & Paul Pierce & 554 & 1452 \\
Nikola Vučević & Scoring Bigs / Two-Way Forwards | Midrange Shot Creators & role_shifting_materially & Bobby Portis, Tobias Harris, T.J. Warren, Jaylen Brown, Jimmy Butler III & Paul Pierce & 1002 & 1904 \\
Kyle O'Quinn & Low-Usage Interior Bigs | Interior Finishers & role_shifting_materially & Lavoy Allen, James Anderson, Stanley Johnson, Xavier Henry, Cory Joseph & Paul Pierce & 283 & 1015 \\
Trae Youn

```{=latex}
\begin{table}[htbp]
\centering
\footnotesize
\resizebox{\textwidth}{!}{%
\begin{tabular}{lllllrr}
\toprule
Player & Hybrid archetype & Drift class & Realistic comps & Ceiling analog & Comp group median points & Comp group median minutes \\
\midrule
Roy 
Hibbert & Scoring Bigs / Two-Way Forwards | Arc-Heavy Spacers & role\_shifting\_materially & Aaron Gordon, Will Barton, Joffrey Lauvergne, Skal Labissiere, Darnell Jackson & Manu Ginobili & 256 & 636.5 \\
Kemba 
Walker & High-Usage Primary Creators | Rim Pressure Drivers & role\_shifting\_materially & Isaiah Thomas, D'Angelo Russell, Marcus Smart, Paul George, Nickeil Alexander-Walker & Paul Pierce & 554 & 1452 \\
Nikola 
Vu\v{c}evi\'c & Scoring Bigs / Two-Way Forwards | Midrange Shot Creators & role\_shifting\_materially & Bobby Portis, Tobias Harris, T.J. Warren, Jaylen Brown, Jimmy Butler III & Paul Pierce & 1002 & 1904 \\
Kyle 
O'Quinn & Low-Usage Interior Bigs | Interior Finishers & role\_shifting\_materially & Lavoy Allen, James Anderson, Stanley Johnson, Xavier Henry, Cory Joseph & Paul Pierce & 283 & 1015 \\
Trae 
Young & High-Usage Primary Creators | Balanced Three-Level Scorers & role\_shifting\_materially & Luka Don\v{c}i\'c, Donovan Mitchell, Shai Gilgeous-Alexander, Jamal Murray, Jordan Poole & Paul Pierce & 1589 & 2154.5 \\
\bottomrule
\end{tabular}%
}
\caption{Compact case-study extraction from the integrated profile output. The table illustrates how current hybrid archetype, drift, realistic comparables, and optional ceiling analogs are returned together at the player level.}
\label{tab:case-profiles}
\end{table}
```